In [2]:
import pandas as pd
import numpy as np
import joblib
from scipy.stats import ks_2samp

before_drift = joblib.load('../data/drift_simulation/before_drift.pkl')
after_drift = joblib.load('../data/drift_simulation/after_drift.pkl')

print(before_drift.shape, after_drift.shape)

(59054, 441) (59054, 441)


In [3]:
model = joblib.load('../models/xgb_fraud_model_v1.pkl')
feature_cols = model.get_booster().feature_names

proba_before = model.predict_proba(before_drift[feature_cols])[:, 1]
proba_after = model.predict_proba(after_drift[feature_cols])[:, 1]

ks_amt = ks_2samp(before_drift['TransactionAmt'], after_drift['TransactionAmt'])
ks_proba = ks_2samp(proba_before, proba_after)

print(f"TransactionAmt KS-test: statistic={ks_amt.statistic:.4f}, p-value={ks_amt.pvalue:.6f}")
print(f"Prediction confidence KS-test: statistic={ks_proba.statistic:.4f}, p-value={ks_proba.pvalue:.6f}")

TransactionAmt KS-test: statistic=0.0262, p-value=0.000000
Prediction confidence KS-test: statistic=0.0250, p-value=0.000000


In [4]:
def detect_drift(reference_proba, window_proba, threshold=0.05):
    """Returns True if drift detected (KS statistic exceeds threshold)."""
    stat, pvalue = ks_2samp(reference_proba, window_proba)
    drifted = stat > threshold and pvalue < 0.01
    return drifted, stat, pvalue

reference_proba = proba_before  # baseline: known-good period

window_size = 5000
results = []

full_stream = np.concatenate([proba_before, proba_after])

for start in range(0, len(full_stream) - window_size, window_size):
    window = full_stream[start:start+window_size]
    drifted, stat, pval = detect_drift(reference_proba, window)
    results.append({'window_start': start, 'ks_stat': stat, 'p_value': pval, 'drift_detected': drifted})

results_df = pd.DataFrame(results)
print(results_df)
print(f"\nDrift injection point was at row: {len(proba_before)}")

    window_start   ks_stat       p_value  drift_detected
0              0  0.040126  6.923106e-07           False
1           5000  0.037680  4.012289e-06           False
2          10000  0.025793  4.258797e-03           False
3          15000  0.030244  4.257797e-04           False
4          20000  0.026803  2.608077e-03           False
5          25000  0.063935  7.955686e-17            True
6          30000  0.049355  3.374484e-10           False
7          35000  0.013397  3.762813e-01           False
8          40000  0.033666  5.647873e-05           False
9          45000  0.032826  9.460956e-05           False
10         50000  0.017142  1.316102e-01           False
11         55000  0.060107  6.401945e-15            True
12         60000  0.066679  2.896369e-18            True
13         65000  0.068494  3.000250e-19            True
14         70000  0.018213  9.276533e-02           False
15         75000  0.054845  1.710965e-12            True
16         80000  0.066740  2.6

In [5]:
for threshold in [0.05, 0.06, 0.07]:
    results_v2 = []
    for start in range(0, len(full_stream) - window_size, window_size):
        window = full_stream[start:start+window_size]
        drifted, stat, pval = detect_drift(reference_proba, window, threshold=threshold)
        results_v2.append({'window_start': start, 'ks_stat': stat, 'drift_detected': drifted})
    
    df_v2 = pd.DataFrame(results_v2)
    false_positives = df_v2[(df_v2['window_start'] < 59054) & (df_v2['drift_detected'])].shape[0]
    true_positives = df_v2[(df_v2['window_start'] >= 55000) & (df_v2['drift_detected'])].shape[0]
    
    print(f"Threshold {threshold}: false positives (before injection) = {false_positives}, detections at/after injection = {true_positives}")

Threshold 0.05: false positives (before injection) = 2, detections at/after injection = 7
Threshold 0.06: false positives (before injection) = 2, detections at/after injection = 6
Threshold 0.07: false positives (before injection) = 0, detections at/after injection = 2


In [6]:
def check_and_trigger_retraining(reference_proba, window_proba, threshold=0.05, consecutive_required=2):
    """
    Returns drift status for a single window. In production, retraining fires
    only after 'consecutive_required' consecutive drifted windows, to avoid
    reacting to a single noisy window.
    """
    drifted, stat, pval = detect_drift(reference_proba, window_proba, threshold)
    return drifted, stat, pval

consecutive_count = 0
consecutive_required = 2
retrain_triggered_at = None

for start in range(0, len(full_stream) - window_size, window_size):
    window = full_stream[start:start+window_size]
    drifted, stat, pval = check_and_trigger_retraining(reference_proba, window, threshold=0.05)
    
    if drifted:
        consecutive_count += 1
    else:
        consecutive_count = 0
    
    if consecutive_count >= consecutive_required and retrain_triggered_at is None:
        retrain_triggered_at = start
        print(f"RETRAINING TRIGGERED at window_start={start} (after {consecutive_required} consecutive drifted windows)")

print(f"\nActual injection point: row 59054")
print(f"Retraining triggered at: row {retrain_triggered_at}")
print(f"Detection lag: {retrain_triggered_at - 59054 if retrain_triggered_at else 'N/A'} rows")

RETRAINING TRIGGERED at window_start=60000 (after 2 consecutive drifted windows)

Actual injection point: row 59054
Retraining triggered at: row 60000
Detection lag: 946 rows


In [7]:
import xgboost as xgb
import time

exclude_cols = ['TransactionID', 'isFraud', 'TransactionDT', 'card_entity']
retrain_feature_cols = [c for c in after_drift.columns if c not in exclude_cols]

# Split the drift segment itself: retrain on first 70%, validate on held-out last 30%
after_drift_sorted = after_drift.sort_values('TransactionDT').reset_index(drop=True)
retrain_split = int(len(after_drift_sorted) * 0.7)

retrain_data = after_drift_sorted.iloc[:retrain_split]
holdout_data = after_drift_sorted.iloc[retrain_split:]

X_retrain = retrain_data[retrain_feature_cols]
y_retrain = retrain_data['isFraud']
X_holdout = holdout_data[retrain_feature_cols]
y_holdout = holdout_data['isFraud']

print(f"Retrain: {X_retrain.shape}, fraud rate: {y_retrain.mean():.4f}")
print(f"Holdout: {X_holdout.shape}, fraud rate: {y_holdout.mean():.4f}")

scale_pos_weight_retrain = (y_retrain == 0).sum() / (y_retrain == 1).sum()

retrained_model = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    scale_pos_weight=scale_pos_weight_retrain,
    eval_metric='auc',
    random_state=42,
    n_jobs=-1
)

start = time.time()
retrained_model.fit(X_retrain, y_retrain)
print(f"Retraining time: {time.time() - start:.1f}s")

Retrain: (41337, 437), fraud rate: 0.0924
Holdout: (17717, 437), fraud rate: 0.0966


Retraining time: 5.2s


In [8]:
from sklearn.metrics import roc_auc_score, precision_score, recall_score

# Old (production) model's performance on the SAME holdout set
old_model_proba_holdout = model.predict_proba(X_holdout)[:, 1]
old_model_auc = roc_auc_score(y_holdout, old_model_proba_holdout)

# Retrained model's performance on the held-out validation slice (never seen during retraining)
new_proba_holdout = retrained_model.predict_proba(X_holdout)[:, 1]
new_pred_holdout = retrained_model.predict(X_holdout)
new_model_auc = roc_auc_score(y_holdout, new_proba_holdout)

print(f"Old production model AUC (on holdout): {old_model_auc:.4f}")
print(f"Retrained model AUC (on holdout):       {new_model_auc:.4f}")
print(f"Improvement: {new_model_auc - old_model_auc:+.4f}")

promotion_threshold = 0.02
should_promote = (new_model_auc - old_model_auc) >= promotion_threshold
print(f"\nShould promote retrained model? {should_promote}")

Old production model AUC (on holdout): 0.7007
Retrained model AUC (on holdout):       0.9441
Improvement: +0.2434

Should promote retrained model? True


In [9]:
old_pred_holdout = model.predict(X_holdout)
new_pred_holdout = retrained_model.predict(X_holdout)

print("--- Old model on holdout ---")
print(f"Precision: {precision_score(y_holdout, old_pred_holdout):.4f}, Recall: {recall_score(y_holdout, old_pred_holdout):.4f}")

print("\n--- Retrained model on holdout ---")
print(f"Precision: {precision_score(y_holdout, new_pred_holdout):.4f}, Recall: {recall_score(y_holdout, new_pred_holdout):.4f}")

--- Old model on holdout ---
Precision: 0.2971, Recall: 0.3834

--- Retrained model on holdout ---
Precision: 0.5034, Recall: 0.7890


In [10]:
import json
from datetime import datetime, timezone

retrain_history_entry = {
    'timestamp': datetime.now(timezone.utc).isoformat(),
    'trigger_reason': 'drift_detected',
    'drift_detection_row': 60000,
    'actual_injection_row': 59054,
    'detection_lag_rows': 946,
    'retrain_set_size': int(len(X_retrain)),
    'holdout_set_size': int(len(X_holdout)),
    'old_model_auc_holdout': float(old_model_auc),
    'new_model_auc_holdout': float(new_model_auc),
    'old_model_precision_holdout': float(precision_score(y_holdout, old_pred_holdout)),
    'old_model_recall_holdout': float(recall_score(y_holdout, old_pred_holdout)),
    'new_model_precision_holdout': float(precision_score(y_holdout, new_pred_holdout)),
    'new_model_recall_holdout': float(recall_score(y_holdout, new_pred_holdout)),
    'promoted': bool(should_promote),
}

history_path = '../models/retraining_history.json'
try:
    with open(history_path, 'r') as f:
        history = json.load(f)
except FileNotFoundError:
    history = []

history.append(retrain_history_entry)

with open(history_path, 'w') as f:
    json.dump(history, f, indent=2)

if should_promote:
    joblib.dump(retrained_model, '../models/xgb_fraud_model_v2_retrained.pkl')
    print("New model saved as v2 (promoted, validated on genuine holdout).")

print(json.dumps(retrain_history_entry, indent=2))

New model saved as v2 (promoted, validated on genuine holdout).
{
  "timestamp": "2026-09-30T11:01:39.485637+00:00",
  "trigger_reason": "drift_detected",
  "drift_detection_row": 60000,
  "actual_injection_row": 59054,
  "detection_lag_rows": 946,
  "retrain_set_size": 41337,
  "holdout_set_size": 17717,
  "old_model_auc_holdout": 0.7006621676719272,
  "new_model_auc_holdout": 0.9440542569768364,
  "old_model_precision_holdout": 0.2971014492753623,
  "old_model_recall_holdout": 0.38340151957919344,
  "new_model_precision_holdout": 0.5033557046979866,
  "new_model_recall_holdout": 0.7890122735242548,
  "promoted": true
}


In [11]:
from sklearn.metrics import precision_recall_curve
import numpy as np

val_df_v2 = joblib.load('../data/processed/val_processed_v2.pkl')
feature_cols_v2 = retrained_model.get_booster().feature_names

X_val_full = val_df_v2[feature_cols_v2]
y_val_full = val_df_v2['isFraud']

proba_v2 = retrained_model.predict_proba(X_val_full)[:, 1]

precisions, recalls, thresholds = precision_recall_curve(y_val_full, proba_v2)

f2_scores = 5 * (precisions * recalls) / (4 * precisions + recalls + 1e-10)
best_idx = np.argmax(f2_scores)

print(f"Best F2 threshold for v2 (retrained) model: {thresholds[best_idx]:.4f}")
print(f"At this threshold -> Precision: {precisions[best_idx]:.4f}, Recall: {recalls[best_idx]:.4f}, F2: {f2_scores[best_idx]:.4f}")

Best F2 threshold for v2 (retrained) model: 0.6801
At this threshold -> Precision: 0.3416, Recall: 0.3952, F2: 0.3832


In [12]:
model_v1 = joblib.load('../models/xgb_fraud_model_v1.pkl')
proba_v1 = model_v1.predict_proba(X_val_full)[:, 1]

precisions_v1, recalls_v1, thresholds_v1 = precision_recall_curve(y_val_full, proba_v1)
f2_scores_v1 = 5 * (precisions_v1 * recalls_v1) / (4 * precisions_v1 + recalls_v1 + 1e-10)
best_idx_v1 = np.argmax(f2_scores_v1)

print(f"Best F2 threshold for v1 (original baseline): {thresholds_v1[best_idx_v1]:.4f}")
print(f"At this threshold -> Precision: {precisions_v1[best_idx_v1]:.4f}, Recall: {recalls_v1[best_idx_v1]:.4f}, F2: {f2_scores_v1[best_idx_v1]:.4f}")

Best F2 threshold for v1 (original baseline): 0.6210
At this threshold -> Precision: 0.3133, Recall: 0.6398, F2: 0.5294
